In [4]:
# Method 1: Your original cluster calculation (from NetCDF)
import xarray as xr

nc_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc'
ds = xr.open_dataset(nc_file)

# Calculate SMB from volume change
volume_2000 = ds['volume'].sel(time=2000).sum()
volume_2020 = ds['volume'].sel(time=2020).sum()
area_2000 = ds['area'].sel(time=2000).sum()

smb_from_netcdf = float((volume_2020 - volume_2000) / area_2000 / 20 * 1000)  # mm/yr

print(f"\n{'='*70}")
print("METHOD COMPARISON:")
print(f"{'='*70}")
print(f"Method 1 (NetCDF volume change): {smb_from_netcdf:.1f} mm/yr")
print(f"Method 2 (Per-glacier MB timeseries): {df_comp['SMB'].mean():.1f} mm/yr (unweighted)")

# Method 3: Area-weighted per-glacier
area_weighted_smb = np.average(df_comp['SMB'], weights=df_comp['area'])
print(f"Method 3 (Area-weighted per-glacier): {area_weighted_smb:.1f} mm/yr")

# GMB
gmb_value = np.average(df_comp['GMB'], weights=df_comp['area'])
print(f"\nGMB (area-weighted): {gmb_value:.1f} mm/yr")

# Calculate biases
print(f"\n{'='*70}")
print("BIASES:")
print(f"{'='*70}")
print(f"Method 1 bias: {smb_from_netcdf - gmb_value:.1f} mm/yr")
print(f"Method 2 bias: {df_comp['SMB'].mean() - gmb_value:.1f} mm/yr (unweighted)")
print(f"Method 3 bias: {area_weighted_smb - gmb_value:.1f} mm/yr (area-weighted)")
print(f"\nExpected bias from cluster comparison: +4.9 mm/yr")
print(f"{'='*70}")

ds.close()


METHOD COMPARISON:
Method 1 (NetCDF volume change): -93.8 mm/yr
Method 2 (Per-glacier MB timeseries): -45.7 mm/yr (unweighted)
Method 3 (Area-weighted per-glacier): -96.1 mm/yr

GMB (area-weighted): -98.7 mm/yr

BIASES:
Method 1 bias: 4.9 mm/yr
Method 2 bias: 53.0 mm/yr (unweighted)
Method 3 bias: 2.7 mm/yr (area-weighted)

Expected bias from cluster comparison: +4.9 mm/yr


In [5]:
# Investigate the mb.csv structure
import pandas as pd

mb_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/mb.csv'
mb_df = pd.read_csv(mb_file)

print("mb.csv structure:")
print(f"  Columns: {list(mb_df.columns)}")
print(f"  Shape: {mb_df.shape}")
print(f"\nFirst few rows:")
print(mb_df.head())

# Check years covered
numeric_cols = mb_df.select_dtypes(include=[np.number]).columns
print(f"\nNumeric columns (years?): {list(numeric_cols)}")
print(f"Number of years: {len(numeric_cols)}")

# Check if it's 2000-2021 or 2000-2020
print(f"\nFirst glacier mean MB: {mb_df.iloc[0][numeric_cols].mean():.1f} mm/yr")

mb.csv structure:
  Columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', 'rgi_id']
  Shape: (422, 23)

First few rows:
            0           1           2           3           4           5  \
0  219.532693 -412.006174  517.612683 -344.317203 -407.416737 -280.290713   
1  338.657394  -16.782901  795.384023 -104.907805  -70.931335  236.883793   
2  272.334399  -67.289462  737.790816 -171.249381 -117.172316  195.502143   
3   11.902746 -275.515940  491.312188 -498.581932 -390.012579  -22.830379   
4  150.933820 -278.268727  566.348844 -350.729536 -329.195641  -32.009636   

            6           7           8           9  ...          13  \
0 -491.618314   14.481736   92.377109 -418.798902  ... -681.449283   
1 -109.254984  219.657793  284.917036 -208.816446  ... -214.219316   
2 -170.986441  163.076412  217.992244 -270.540938  ... -253.967077   
3 -440.770339 -150.624990  -73.320652 -529.454350  ... -452

In [6]:
# Correct per-glacier analysis (using NetCDF - same method as cluster)
import xarray as xr
import pandas as pd
import numpy as np
from oggm import utils

nc_file = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/Output/CR2MET/DA1/run_output_2000_2019_hydro_TC_DA1.nc'
ds = xr.open_dataset(nc_file)

# Load geodetic data
geodetic_ref = utils.get_geodetic_mb_dataframe()
geodetic_ref = geodetic_ref[geodetic_ref['period'] == '2000-01-01_2020-01-01']

# Calculate per-glacier SMB (same method as cluster)
glacier_ids = [str(x) for x in ds['rgi_id'].values]
comparison = []

for i, rgi_id in enumerate(glacier_ids):
    if rgi_id not in geodetic_ref.index:
        continue
    
    # Extract this glacier's data
    vol_2000 = float(ds['volume'].sel(time=2000, rgi_id=rgi_id).values)
    vol_2020 = float(ds['volume'].sel(time=2020, rgi_id=rgi_id).values)
    area_2000 = float(ds['area'].sel(time=2000, rgi_id=rgi_id).values)
    
    # Calculate SMB (same as cluster method)
    smb = ((vol_2020 - vol_2000) / area_2000 / 20) * 1000  # mm/yr
    
    # Get GMB
    gmb = geodetic_ref.loc[rgi_id, 'dmdtda'] * 1000  # mm/yr
    gmb_area = geodetic_ref.loc[rgi_id, 'area']
    
    comparison.append({
        'rgi_id': rgi_id,
        'SMB': smb,
        'GMB': gmb,
        'delta': smb - gmb,
        'abs_delta': abs(smb - gmb),
        'area': gmb_area
    })

df_comp = pd.DataFrame(comparison)

# Now calculate area-weighted stats
weighted_mean_delta = np.average(df_comp['delta'], weights=df_comp['area'])

print(f"\n{'='*70}")
print("CORRECT Per-Glacier Analysis (using NetCDF method):")
print(f"{'='*70}")
print(f"Total glaciers: {len(df_comp)}")
print(f"Area-weighted mean delta: {weighted_mean_delta:.1f} mm/yr")
print(f"Expected (cluster-level): +4.9 mm/yr")
print(f"{'='*70}")

print(f"\nWorst 10 glaciers:")
print(df_comp.nlargest(10, 'abs_delta')[['rgi_id', 'SMB', 'GMB', 'delta', 'area']])


CORRECT Per-Glacier Analysis (using NetCDF method):
Total glaciers: 422
Area-weighted mean delta: 2.7 mm/yr
Expected (cluster-level): +4.9 mm/yr

Worst 10 glaciers:
             rgi_id         SMB         GMB       delta     area
374  RGI60-17.15496  418.053012   50.300000  367.753012  28000.0
98   RGI60-17.14996  501.463569  286.800000  214.663569  49000.0
9    RGI60-17.14635 -214.055637 -426.700000  212.644363  41000.0
381  RGI60-17.15508  -30.047916 -220.802621  190.754704  12000.0
3    RGI60-17.14559 -145.506156 -330.900000  185.393844  31000.0
209  RGI60-17.15593  154.919176  -25.700000  180.619176  44000.0
359  RGI60-17.15467  361.277090  193.800000  167.477090  43000.0
239  RGI60-17.15618  -74.090721 -220.802621  146.711899  11000.0
166  RGI60-17.15145  -79.217061 -220.802621  141.585560  11000.0
379  RGI60-17.15509  206.674208   67.600000  139.074208  73000.0


In [3]:
# Investigate the worst offenders
print("\nTop 5 problem glaciers (detailed):")
worst = df_comp.nlargest(5, 'abs_delta')
for _, row in worst.iterrows():
    print(f"\n{row['rgi_id']}:")
    print(f"  SMB: {row['SMB']:+.1f} mm/yr")
    print(f"  GMB: {row['GMB']:+.1f} mm/yr")
    print(f"  Bias: {row['delta']:+.1f} mm/yr ({row['delta']/row['GMB']*100:+.0f}%)")
    print(f"  Area: {row['area']/1e6:.3f} km²")


Top 5 problem glaciers (detailed):

RGI60-17.15496:
  SMB: +418.1 mm/yr
  GMB: +50.3 mm/yr
  Bias: +367.8 mm/yr (+731%)
  Area: 0.028 km²

RGI60-17.14996:
  SMB: +501.5 mm/yr
  GMB: +286.8 mm/yr
  Bias: +214.7 mm/yr (+75%)
  Area: 0.049 km²

RGI60-17.14635:
  SMB: -214.1 mm/yr
  GMB: -426.7 mm/yr
  Bias: +212.6 mm/yr (-50%)
  Area: 0.041 km²

RGI60-17.15508:
  SMB: -30.0 mm/yr
  GMB: -220.8 mm/yr
  Bias: +190.8 mm/yr (-86%)
  Area: 0.012 km²

RGI60-17.14559:
  SMB: -145.5 mm/yr
  GMB: -330.9 mm/yr
  Bias: +185.4 mm/yr (-56%)
  Area: 0.031 km²
